In [9]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [10]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [11]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)

In [12]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [13]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [14]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [15]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [17]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_s3_region(s3_url: str) -> str:\n    \"\"\"\n    Extract AWS region from an S3 bucket URL.\n    \n    Supports multiple S3 URL formats:\n    - https://s3.us-west-2.amazonaws.com/my-bucket\n    - https://s3.amazonaws.com/my-bucket (returns 'us-east-1')\n    - https://my-bucket.s3.us-west-2.amazonaws.com\n    - https://my-bucket.s3.amazonaws.com (returns 'us-east-1')\n    \n    Args:\n        s3_url: The S3 bucket URL\n        \n    Returns:\n        The AWS region string\n        \n    Raises:\n        ValueError: If the URL format is invalid or region cannot be extracted\n    \"\"\"\n    if not s3_url:\n        raise ValueError(\"S3 URL cannot be empty\")\n    \n    # Pattern 1: s3.region.amazonaws.com/bucket format\n    match = re.search(r's3\\.([a-z0-9\\-]+)\\.amazonaws\\.com', s3_url)\n    if ma

In [8]:
dataset = generate_dataset()
print(dataset)

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URL. For example, 'https://s3.us-west-2.amazonaws.com/my-bucket' should return 'us-west-2'."}, {'task': "Create a JSON CloudFormation template snippet that defines an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'."}, {'task': "Write a regular expression that validates AWS IAM role ARNs in the format 'arn:aws:iam::123456789012:role/RoleName'."}]
